<img src="../_static/pymt-logo-header-text.png">

## Coastline Evolution Model + Waves

* Link to this notebook: https://github.com/csdms/pymt/blob/master/notebooks/cem_and_waves.ipynb
* Install command: `$ conda install notebook pymt_cem`

This example explores how to use a BMI implementation to couple the Waves component with the Coastline Evolution Model component.

### Links
* [CEM source code](https://github.com/csdms/cem-old): Look at the files that have *deltas* in their name.
* [CEM description on CSDMS](http://csdms.colorado.edu/wiki/Model_help:CEM): Detailed information on the CEM model.

### Interacting with the Coastline Evolution Model BMI using Python

Some magic that allows us to view images within the notebook.

In [204]:
%matplotlib inline
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

%load_ext autoreload
%autoreload 2

import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Import the `Cem` class, and instantiate it. In Python, a model with a BMI will have no arguments for its constructor. Note that although the class has been instantiated, it's not yet ready to be run. We'll get to that later!

In [205]:
from pymt import models
from src.test_compose import compose
from src.test_compose import composeConfig

In [206]:
cem, waves = models.Cem(), models.Waves()

cem_compose = compose(waves,cem)

Even though we can't run our waves model yet, we can still get some information about it. *Just don't try to run it.* Some things we can do with our model are get the names of the input variables.

In [207]:
waves.get_input_var_names()


('sea_surface_water_wave__height',
 'sea_surface_water_wave__period',
 'sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_highness_parameter',
 'sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_asymmetry_parameter')

In [208]:
cem.get_input_var_names()


('sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity',
 'land_surface_water_sediment~bedload__mass_flow_rate',
 'sea_surface_water_wave__period',
 'sea_surface_water_wave__height',
 'land_surface__elevation',
 'model__time_step')

We can also get information about specific variables. Here we'll look at some info about wave direction. This is the main input of the Cem model. Notice that BMI components always use [CSDMS standard names](http://csdms.colorado.edu/wiki/CSDMS_Standard_Names). The CSDMS Standard Name for wave angle is,

    "sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"

Quite a mouthful, I know. With that name we can get information about that variable and the grid that it is on (it's actually not a one).

OK. We're finally ready to run the model. Well not quite. First we initialize the model with the BMI **initialize** method. Normally we would pass it a string that represents the name of an input file. For this example we'll pass **None**, which tells Cem to use some defaults.

In [209]:




args = cem_compose.setup()
print(args)
cem_compose.initialize(args)

args = cem.setup(number_of_rows=100, number_of_cols=200, grid_spacing=200.0)
cem.initialize(*args)

print(waves.get_value("sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"
))
print(cem.get_value("sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"))



('waves.txt', '/tmp/tmpnpq76aqh')
('mergedConf.txt', '.')
Condition Initial 
Condition Initial 
[-0.52460497]
[0.]


Setting end time to 3650
CEM: trying to open file: bmi2Conf.txt
CEM: line:  50, 1000, 1000.0, 1

CEM: number of rows, columns: 50, 1000
*** Grid size is (0,0)
*** Requested size is (50,2000)
*** New grid size is (50,2000)
CEM: trying to open file: cem.txt
CEM: line: 100, 200, 200.0, 1

CEM: number of rows, columns: 100, 200
*** Grid size is (0,0)
*** Requested size is (100,400)
*** New grid size is (100,400)


In [210]:

def intersection(x,y):
  return [i for i in x if i in y]

interfaceVars = intersection(waves.get_output_var_names(), cem.get_input_var_names())

print(interfaceVars)


['sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity', 'sea_surface_water_wave__height', 'sea_surface_water_wave__period']


Here I define a convenience function for plotting the water depth and making it look pretty. You don't need to worry too much about it's internals for this tutorial. It just saves us some typing later on.

In [211]:
def plot_coast(spacing, z):
    import matplotlib.pyplot as plt

    xmin, xmax = 0.0, z.shape[1] * spacing[1] * 1e-3
    ymin, ymax = 0.0, z.shape[0] * spacing[0] * 1e-3

    plt.imshow(z, extent=[xmin, xmax, ymin, ymax], origin="lower", cmap="ocean")
    plt.colorbar().ax.set_ylabel("Water Depth (m)")
    plt.xlabel("Along shore (km)")
    plt.ylabel("Cross shore (km)")


It generates plots that look like this. We begin with a flat delta (green) and a linear coastline (y = 3 km). The bathymetry drops off linearly to the top of the domain.

In [212]:
grid_id = cem.var_grid("sea_water__depth")
spacing = cem.grid_spacing(grid_id)
shape = cem.grid_shape(grid_id)
z = np.empty(shape)
cem_compose.get_value("sea_water__depth", z)
plot_coast(spacing, z)

Allocate memory for the sediment discharge array and set the discharge at the coastal cell to some value.

In [213]:
qs = np.zeros_like(z)
qs[0, 100] = 750

The CSDMS Standard Name for this variable is:

    "land_surface_water_sediment~bedload__mass_flow_rate"

You can get an idea of the units based on the quantity part of the name. "mass_flow_rate" indicates mass per time. You can double-check this with the BMI method function **get_var_units**.

In [214]:
cem.get_var_units("land_surface_water_sediment~bedload__mass_flow_rate")

/tmp/ipykernel_133034/2188898298.py:1: DeprecationWarning: Call to deprecated method get_var_units. (use var_units)
  cem.get_var_units("land_surface_water_sediment~bedload__mass_flow_rate")


'kg / s'

In [215]:
cem_compose.set_value(
    "sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_asymmetry_parameter",
    0.3,
)
cem_compose.set_value(
    "sea_shoreline_wave~incoming~deepwater__ashton_et_al_approach_angle_highness_parameter",
    0.7,
)
print(waves.get_value("sea_surface_water_wave__period"))

cem_compose.set_value("sea_surface_water_wave__height", 2.0)
cem_compose.set_value("sea_surface_water_wave__period", 7.0)

cem_compose.get_value("sea_surface_water_wave__height")

print(waves.get_value("sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"))



[7.]
[-0.52460497]


Set the bedload flux and run the model.

In [216]:

qs = np.zeros_like(z)
qs[0, 100] = 750


for time in range(3000):
    print(waves.get_value("sea_surface_water_wave__azimuth_angle_of_opposite_of_phase_velocity"))
    cem.set_value("land_surface_water_sediment~bedload__mass_flow_rate", qs)
    cem_compose.update()

    
cem_compose.get_value("sea_water__depth", z)


[-0.52460497]
==== WaveAngle: -27.56:  MASS Percent: 0.0000:  Time Step: 0
[-0.48103347]
[1.38764202]
[-0.14330953]
[1.22210285]
[-1.21419877]
[0.04342708]
[-1.02299474]
[1.07996482]
[0.84048507]
[-1.36152079]
[-1.27807636]
[-1.34213664]
[-0.15841837]
[1.53761068]
[-1.07929515]
[-0.50851508]
[-0.20998747]
[1.53567177]
[-0.0332102]
[-1.27516505]
[0.37793707]
[-0.9622934]
[0.27570596]
[-0.90794564]
[-0.96986708]
[-0.84800478]
[-1.06819193]
[-1.21559644]
[-1.07973093]
[-1.47541464]
[-1.37795013]
[0.92030744]
[1.51274201]
[-1.08716367]
[-0.30292564]
[-0.48547905]
[0.23412834]
[0.1224276]
[1.11527488]
[0.99536459]
[1.55481155]
[-0.46555072]
[-0.31244276]
[-1.56530501]
[-0.01713085]
[1.01140523]
[-0.09405645]
[-0.61942596]
[-1.43271565]
[-0.20981835]
[0.26457374]
[-0.08997453]
[-0.45873219]
[-1.39886617]
[-1.55635734]
[-0.83605977]
[-0.174552]
[-0.65438569]
[-0.86760999]
[-1.02299065]
[-1.5665895]
[1.38501782]
[-0.25023743]
[-0.24743168]
[-0.90700229]
[0.98692124]
[-0.87131071]
[-0.58936863]

array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [217]:
plot_coast(spacing, z)

Let's add another sediment source with a different flux and update the model.

In [218]:
qs[0, 150] = 500
for time in range(3750):
    cem.set_value("land_surface_water_sediment~bedload__mass_flow_rate", qs)
    cem_compose.update()

cem_compose.get_value("sea_water__depth", z)

==== WaveAngle: -29.12:  MASS Percent: 1.0084:  Time Step: 3000
==== WaveAngle: -36.58:  MASS Percent: 1.0132:  Time Step: 4000
==== WaveAngle: -50.66:  MASS Percent: 1.0179:  Time Step: 5000
==== WaveAngle: 69.97:  MASS Percent: 1.0224:  Time Step: 6000


array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [219]:
plot_coast(spacing, z)

Here we shut off the sediment supply completely.

In [220]:
qs.fill(0.0)
for time in range(4000):
    cem.set_value("land_surface_water_sediment~bedload__mass_flow_rate", qs)
    cem_compose.update()

cem_compose.get_value("sea_water__depth", z)

==== WaveAngle: -81.98:  MASS Percent: 1.0253:  Time Step: 7000
==== WaveAngle: 89.55:  MASS Percent: 1.0251:  Time Step: 8000
==== WaveAngle: 59.77:  MASS Percent: 1.0249:  Time Step: 9000
==== WaveAngle: -90.00:  MASS Percent: 1.0247:  Time Step: 10000


array([[-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       [-1. , -1. , -1. , ..., -1. , -1. , -1. ],
       ...,
       [22.4, 22.4, 22.4, ..., 22.4, 22.4, 22.4],
       [22.6, 22.6, 22.6, ..., 22.6, 22.6, 22.6],
       [22.8, 22.8, 22.8, ..., 22.8, 22.8, 22.8]])

In [221]:
plot_coast(spacing, z)